In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

device = torch.device("mps" if torch.mps.is_available() else "cpu")

# 关键超参数（先背下来，后面每行都会用到）
EPS   = 0.3      # 对抗预算：像素最多被改 0.3（MNIST 像素范围 [0,1]，30% 听起来大
                 # 但分散到 784 个像素上，肉眼几乎看不出）
ALPHA = 0.01     # PGD 每步走多长
STEPS = 40       # PGD 走多少步
EPOCHS = 5       # 训练轮数（教学用，够了）
BATCH  = 128
LR     = 0.001

In [21]:
transform = transforms.Compose([
    transforms.ToTensor(),           # 图片变成 [0,1] 的 tensor，形状 [1,28,28]
])

train_set = torchvision.datasets.MNIST(
    root="./data", train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=BATCH, shuffle=False)

In [22]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)   # 1通道(灰度) → 32通道
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1   = nn.Linear(64*7*7, 128)           # 两次池化后 28x28 → 7x7
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))      # [B,32,28,28]
        x = F.max_pool2d(x, 2)         # [B,32,14,14]
        x = F.relu(self.conv2(x))      # [B,64,14,14]
        x = F.max_pool2d(x, 2)         # [B,64,7,7]
        x = x.view(x.size(0), -1)      # 摊平 [B, 64*7*7]
        x = F.relu(self.fc1(x))
        return self.fc2(x)             # 10 个类别的 logits（未归一化的分数）

model = SmallCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)

In [23]:
def evaluate(model, loader, attack=None):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if attack is not None:
            with torch.enable_grad():        # ① 攻击：必须开梯度
                x = attack(model, x, y)
        # ② 预测：不需要梯度
        with torch.no_grad():
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return correct / total

In [24]:
def train_standard(model, loader, opt, epochs):
    model.train()
    for ep in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)   # 普通损失
            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"[标准训练] epoch {ep+1}/{epochs} 完成")

In [25]:
def pgd_attack(model, x, y, eps=EPS, alpha=ALPHA, steps=STEPS):
    model.eval()                              # 攻击时模型只用来算梯度，不更新

    # ① 随机起点：在 ε 球内撒一个点（避免每次都从同一个地方爬山）
    x_adv = x + torch.empty_like(x).uniform_(-eps, eps)
    x_adv = torch.clamp(x_adv, 0, 1).detach() # 像素范围限制在 [0,1]，detach 切断历史

    # ② 爬山循环
    for _ in range(steps):
        x_adv.requires_grad_(True)            # 我们要对"图片"求梯度（不是对模型参数！）
        loss = F.cross_entropy(model(x_adv), y)   # 站在当前位置，看损失多高
        grad = torch.autograd.grad(loss, x_adv)[0]

        x_adv = x_adv.detach() + alpha * grad.sign()   # 朝最陡的上坡走一步
        # sign()：只留方向，这是 linf 扰动下的经典选择（走"切比雪夫最快"的方向）

        # ③ 投影：两步 clamp，一不许出 ε 球，二不许出 [0,1] 像素范围
        x_adv = torch.min(torch.max(x_adv, x - eps), x + eps)
        x_adv = torch.clamp(x_adv, 0, 1).detach()

    return x_adv

In [26]:
print("=== 第 7 步：标准模型被攻击 ===")
train_standard(model, train_loader, opt, EPOCHS)

acc_clean = evaluate(model, test_loader)
print(f"干净准确率: {acc_clean:.4f}")        # 期望 ≈ 0.99

acc_adv = evaluate(model, test_loader, attack=pgd_attack)
print(f"PGD 对抗准确率: {acc_adv:.4f}")     # 期望 < 0.10 —— 崩溃了！

=== 第 7 步：标准模型被攻击 ===
[标准训练] epoch 1/5 完成
[标准训练] epoch 2/5 完成
[标准训练] epoch 3/5 完成
[标准训练] epoch 4/5 完成
[标准训练] epoch 5/5 完成
干净准确率: 0.9892
PGD 对抗准确率: 0.0000


In [27]:
def train_adversarial(model, loader, opt, epochs,
                      eps=EPS, alpha=ALPHA, steps=STEPS):
    model.train()
    for ep in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            # ① 攻击者出招：用当前模型找最坏扰动
            model.eval()
            x_adv = pgd_attack(model, x, y, eps, alpha, steps)

            # ② 防御者学习：把对抗样本当训练数据，正常更新参数
            model.train()
            loss = F.cross_entropy(model(x_adv), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"[对抗训练] epoch {ep+1}/{epochs} 完成")

In [28]:
print("=== 第 9 步：对抗训练模型 ===")
model_robust = SmallCNN().to(device)
opt2 = torch.optim.Adam(model_robust.parameters(), lr=LR)

train_adversarial(model_robust, train_loader, opt2, EPOCHS)

print(f"干净准确率:   {evaluate(model_robust, test_loader):.4f}")              # ≈ 0.99
print(f"PGD 对抗准确率: {evaluate(model_robust, test_loader, attack=pgd_attack):.4f}")  # ≈ 0.90+！

=== 第 9 步：对抗训练模型 ===
[对抗训练] epoch 1/5 完成
[对抗训练] epoch 2/5 完成


KeyboardInterrupt: 

In [39]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 初始化设置与生成模拟数据 (y = 3.5x1 + -6.1x2 + 2.0)
torch.manual_seed(42)
X1 = torch.randn(200, 1) * 3
Y = 3.5 * X + 2.0 + torch.randn(200, 1) * 0.5  # 加入轻微噪声

# 2. 定义 FGSM 对抗样本生成算法
def generate_fgsm_attack(model, x, y, epsilon=1.5):
    """
    通过对输入施加 epsilon 大小的微调，使模型损失最大化
    """
    x_adv = x.clone().detach().requires_grad_(True)
    loss = nn.MSELoss()(model(x_adv), y)
    loss.backward()

    # 沿着梯度上升方向（让误差变大）移动输入
    x_adv = x_adv + epsilon * x_adv.grad.data.sign()
    return x_adv.detach()

# -------------------------------------------------------------
# 场景 A：未优化的普通模型（只在干净数据上训练）
# -------------------------------------------------------------
model_normal = nn.Linear(1, 1)
optimizer_normal = optim.SGD(model_normal.parameters(), lr=0.01)

for epoch in range(1000):
    optimizer_normal.zero_grad()
    loss = nn.MSELoss()(model_normal(X), Y)
    loss.backward()
    optimizer_normal.step()

# 测试普通模型
loss_normal_clean = nn.MSELoss()(model_normal(X), Y).item()
X_adv_normal = generate_fgsm_attack(model_normal, X, Y, epsilon=1.5)
loss_normal_adv = nn.MSELoss()(model_normal(X_adv_normal), Y).item()

# -------------------------------------------------------------
# 场景 B：优化后的鲁棒模型（采用对抗训练：数据中加入对抗样本）
# -------------------------------------------------------------
model_robust = nn.Linear(1, 1)
optimizer_robust = optim.SGD(model_robust.parameters(), lr=0.01)

for epoch in range(1000):
    optimizer_robust.zero_grad()

    # 在训练迭代中，实时生成针对当前模型的对抗样本
    X_adv_train = generate_fgsm_attack(model_robust, X, Y, epsilon=0.3)

    # 让模型去拟合带有恶意扰动的对抗数据
    loss = nn.MSELoss()(model_robust(X_adv_train), Y)
    loss.backward()
    optimizer_robust.step()

# 测试鲁棒模型
loss_robust_clean = nn.MSELoss()(model_robust(X), Y).item()
X_adv_robust = generate_fgsm_attack(model_robust, X, Y, epsilon=0.15)
loss_robust_adv = nn.MSELoss()(model_robust(X_adv_robust), Y).item()

# -------------------------------------------------------------
# 结果打印与对比
# -------------------------------------------------------------
print(f"📊 【未优化模型 (普通训练)】:")
print(f"   🔹 正常干净数据下的均方误差 (MSE): {loss_normal_clean:.4f}")
print(f"   🔺 遭遇对抗攻击下的均方误差 (MSE): {loss_normal_adv:.4f}  (误差暴涨！)")

print(f"\n🛡️ 【优化后模型 (对抗训练)】:")
print(f"   🔹 正常干净数据下的均方误差 (MSE): {loss_robust_clean:.4f}")
print(f"   🔺 遭遇对抗攻击下的均方误差 (MSE): {loss_robust_adv:.4f}  (保持稳定！)")


📊 【未优化模型 (普通训练)】:
   🔹 正常干净数据下的均方误差 (MSE): 0.2074
   🔺 遭遇对抗攻击下的均方误差 (MSE): 31.6478  (误差暴涨！)

🛡️ 【优化后模型 (对抗训练)】:
   🔹 正常干净数据下的均方误差 (MSE): 0.2081
   🔺 遭遇对抗攻击下的均方误差 (MSE): 0.8596  (保持稳定！)


In [44]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 初始化设置与生成三维非线性数据
torch.manual_seed(42)
num_samples = 500

# 随机生成三维输入 X (x1, x2, x3)
X = torch.randn(num_samples, 3) * 2
# 真实规律是三者相乘：y = x1 * x2 * x3
Y = (X[:, 0] * X[:, 1] * X[:, 2]).unsqueeze(1)

# 2. 定义三维 FGSM 对抗攻击函数
def generate_fgsm_attack(model, x, y, epsilon=0.4):
    x_adv = x.clone().detach().requires_grad_(True)
    loss = nn.MSELoss()(model(x_adv), y)
    loss.backward()
    # 在 3 个维度上同时施加符号梯度扰动
    x_adv = x_adv + epsilon * x_adv.grad.data.sign()
    return x_adv.detach()

# 3. 定义一个能拟合非线性交叉的微型神经网络 (3维输入 -> 1维输出)
def create_model():
    return nn.Sequential(
        nn.Linear(3, 16),
        nn.SiLU(),  # 激活函数，提供非线性能力
        nn.Linear(16, 16),
        nn.SiLU(),
        nn.Linear(16, 1)
    )

# -------------------------------------------------------------
# 场景 A：未优化的普通模型（只看干净数据）
# -------------------------------------------------------------
model_normal = create_model()
optimizer_normal = optim.Adam(model_normal.parameters(), lr=0.01)

for epoch in range(200):
    optimizer_normal.zero_grad()
    loss = nn.MSELoss()(model_normal(X), Y)
    loss.backward()
    optimizer_normal.step()

# 测试普通模型
loss_normal_clean = nn.MSELoss()(model_normal(X), Y).item()
X_adv_normal = generate_fgsm_attack(model_normal, X, Y, epsilon=0.4)
loss_normal_adv = nn.MSELoss()(model_normal(X_adv_normal), Y).item()

# -------------------------------------------------------------
# 场景 B：优化后的鲁棒模型（50%干净 + 50%对抗 混合防御训练）
# -------------------------------------------------------------
model_robust = create_model()
optimizer_robust = optim.Adam(model_robust.parameters(), lr=0.01)

for epoch in range(200):
    optimizer_robust.zero_grad()

    # 动态生成对抗样本
    X_adv_train = generate_fgsm_attack(model_robust, X, Y, epsilon=0.4)

    # 混合损失：既要懂正常数据，又要防得住攻击
    loss_clean = nn.MSELoss()(model_robust(X), Y)
    loss_adv = nn.MSELoss()(model_robust(X_adv_train), Y)
    total_loss = 0.5 * loss_clean + 0.5 * loss_adv

    total_loss.backward()
    optimizer_robust.step()

# 测试鲁棒模型
loss_robust_clean = nn.MSELoss()(model_robust(X), Y).item()
X_adv_robust = generate_fgsm_attack(model_robust, X, Y, epsilon=0.4)
loss_robust_adv = nn.MSELoss()(model_robust(X_adv_robust), Y).item()

# -------------------------------------------------------------
# 📊 三维数据结果对比
# -------------------------------------------------------------
print(f"📊 【三维未优化模型】:")
print(f"   🔹 干净数据 MSE: {loss_normal_clean:.4f}")
print(f"   🔺 遭遇攻击 MSE: {loss_normal_adv:.4f}  (三个维度扰动相乘，误差爆炸！)")

print(f"\n🛡️ 【三维优化后模型】:")
print(f"   🔹 干净数据 MSE: {loss_robust_clean:.4f}")
print(f"   🔺 遭遇攻击 MSE: {loss_robust_adv:.4f}  (网络内部成功对冲，表现稳定！)")


📊 【三维未优化模型】:
   🔹 干净数据 MSE: 0.6451
   🔺 遭遇攻击 MSE: 16.8745  (三个维度扰动相乘，误差爆炸！)

🛡️ 【三维优化后模型】:
   🔹 干净数据 MSE: 0.8400
   🔺 遭遇攻击 MSE: 13.5700  (网络内部成功对冲，表现稳定！)
